1. Load the dataset

In [ ]:
import pandas as pd

df = pd.read_csv("cleaned_manufacturing_quality.csv")

print(df.head())
print(df.shape)
print(df.info())

    batch    line    shift  quantity machine   age_band  operating_hours  \
0  B00001  Line B  Morning       246     M11   6+ years             6433   
1  B00002  Line B    Night       271     M09  0-2 years             4363   
2  B00003  Line C    Night       157     M07  0-2 years             3552   
3  B00004  Line B  Morning       152     M12   6+ years            17420   
4  B00005  Line C  Morning       177     M12   6+ years            17420   

   temperature  pressure   speed        supplier material_grade      lot  \
0       201.33      6.33  1302.1  Supplier Delta              B  LOT-469   
1       200.99      6.50  1223.8  Supplier Gamma              B  LOT-743   
2       182.06      5.43  1234.4   Supplier Beta              B  LOT-576   
3       201.29      5.59  1157.1  Supplier Alpha              B  LOT-143   
4       198.88      5.56  1271.8  Supplier Delta              A  LOT-890   

       defect_type  defect_count quality_status  defective  
0    Surface Crack       

2. Check missing values and duplicates

In [ ]:
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

batch              0
line               0
shift              0
quantity           0
machine            0
age_band           0
operating_hours    0
temperature        0
pressure           0
speed              0
supplier           0
material_grade     0
lot                0
defect_type        0
defect_count       0
quality_status     0
defective          0
dtype: int64
Duplicate rows: 0


3. Find the defect rate for each production line

In [ ]:
line_eda = df.groupby("line").agg(
    total_defects=("defect_count", "sum"),
    total_quantity=("quantity", "sum")
)

line_eda["defect_rate"] = (
    line_eda["total_defects"] /
    line_eda["total_quantity"] * 100
)

print(line_eda.sort_values(
    "defect_rate", ascending=False
))

        total_defects  total_quantity  defect_rate
line                                              
Line B            779           68238     1.141593
Line D            423           49886     0.847933
Line A            418           49622     0.842368
Line C            464           59228     0.783413


4. Compare defect rates across shifts

In [ ]:
shift_eda = df.groupby("shift").agg(
    total_defects=("defect_count", "sum"),
    total_quantity=("quantity", "sum")
)

shift_eda["defect_rate"] = (
    shift_eda["total_defects"] /
    shift_eda["total_quantity"] * 100
)

print(shift_eda)

           total_defects  total_quantity  defect_rate
shift                                                
Afternoon            714           79407     0.899165
Morning              684           82687     0.827216
Night                686           64880     1.057337


5. Identify the most common defect type

In [ ]:
defect_eda = (
    df.groupby("defect_type")["defect_count"]
      .sum()
      .sort_values(ascending=False)
)

print(defect_eda)

defect_type
Surface Crack      635
Dimension Error    464
Porosity           412
Warping            322
Scratch            251
No Defect            0
Name: defect_count, dtype: int64


6. Analyze suppliers, process parameters and machines

1. Prepare features for machine learning

In [ ]:
supplier_eda = df.groupby("supplier").agg(
    defects=("defect_count", "sum"),
    quantity=("quantity", "sum")
)

supplier_eda["defect_rate"] = (
    supplier_eda["defects"] /
    supplier_eda["quantity"] * 100
)

print(supplier_eda.sort_values(
    "defect_rate", ascending=False
))

                defects  quantity  defect_rate
supplier                                      
Supplier Gamma      646     57579     1.121937
Supplier Beta       595     65126     0.913614
Supplier Delta      377     45414     0.830140
Supplier Alpha      449     55747     0.805425
Unknown              17      3108     0.546976
